# SHAP Explainability

**Question:** *Why did the model approve or reject this particular borrower? Globally — what features drive the model?*

Run `python -m src.train` first so `outputs/model.pkl` exists. Requires `shap`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import pandas as pd
import shap

from src import data_loader, explainer, model as Mdl

trained = Mdl.load_model()
df = data_loader.load_clean().sample(5000, random_state=42).reset_index(drop=True)
X = df[trained.feature_cols].astype(float).fillna(0.0)
bundle = explainer.compute_shap(trained, X, max_samples=2000)
assert bundle is not None, 'SHAP not available — pip install shap'

## Global feature importance (mean |SHAP|)

In [ ]:
explainer.global_importance(bundle).head(15)

In [ ]:
shap.summary_plot(bundle.values, bundle.X, show=False)
plt.savefig('../outputs/figures/shap_summary.png', dpi=140, bbox_inches='tight')
plt.show()

## Top 3 drivers — plain English

In [ ]:
for entry in explainer.top_features_plain_english(bundle, k=3):
    print(f"• {entry['feature']:>18}  (mean |SHAP|={entry['mean_abs_shap']:.4f})\n    {entry['interpretation']}\n")

## Waterfall — one specific borrower

In [ ]:
i = 0
exp = shap.Explanation(values=bundle.values[i],
                       base_values=bundle.expected_value,
                       data=bundle.X.iloc[i],
                       feature_names=bundle.feature_names)
shap.plots.waterfall(exp, show=False)
plt.savefig('../outputs/figures/shap_waterfall.png', dpi=140, bbox_inches='tight')
plt.show()